# Feature Engineering for Pose Data - Workout Monitoring System

## Purpose of this Notebook

**WHY is Feature Engineering Critical for Pose-Based Exercise Recognition?**

### The Challenge with Raw Landmarks

Raw pose landmarks (132-dimensional vectors) have fundamental limitations:

1. **Position Dependency** 🎥
   - Coordinates change with camera angle, distance, and person position
   - Same exercise performed at different screen locations = different raw values
   - Makes model training unstable and generalization poor

2. **Scale Dependency** 📏
   - Absolute coordinates vary with person's size and camera zoom
   - Tall person vs. child = completely different coordinate ranges
   - Model can't distinguish "exercise type" from "person size"

3. **Lack of Semantic Meaning** 🧠
   - x, y, z coordinates don't represent biomechanically meaningful metrics
   - Fitness professionals think in angles, ranges of motion, speeds
   - Raw coordinates obscure the actual movement patterns

4. **High Dimensionality** 📊
   - 132 features per frame × multiple frames = computational overhead
   - Many redundant or correlated features
   - Increases model complexity without adding information

### The Solution: Domain-Specific Feature Engineering

By transforming raw landmarks into biomechanically meaningful features:

✅ **Invariance**: Features remain consistent regardless of camera position/zoom  
✅ **Interpretability**: Angles and velocities have real-world meaning  
✅ **Reduced Dimensionality**: 72 engineered features vs. 132 raw landmarks  
✅ **Better Signal**: Domain knowledge amplifies discriminative patterns  
✅ **Improved Performance**: Models learn faster, generalize better

---

## What We'll Build Today

We'll extract **72 carefully designed features** across 5 categories:

| Category | Count | Description | Why Important |
|----------|-------|-------------|---------------|
| **Angular Features** | 30 | Joint angles (mean, std, range) | Rotation-invariant, biomechanically meaningful |
| **Distance Features** | 20 | Inter-joint distances (normalized) | Scale-invariant body proportions |
| **Velocity Features** | 20 | 1st derivative of motion | Captures movement speed patterns |
| **Smoothness** | 1 | Jerk-based smoothness metric | Distinguishes controlled vs. jerky form |
| **Temporal** | 1 | Sequence length | Exercise tempo information |

Let's get started! 🚀

In [ ]:
# Import libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import sys
import warnings
warnings.filterwarnings('ignore')

# Add src to path
sys.path.append('../src')
from preprocessing.feature_engineering import FeatureEngineer

# Set visualization style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 10
%matplotlib inline

print("✓ Libraries imported")
print("✓ FeatureEngineer class loaded")

## 2. Load Processed Pose Data

**WHY load from processed data?**
- We need clean, normalized pose sequences
- Previous pipeline has handled: video processing → pose extraction → normalization
- Now we transform landmarks → engineered features

**Data Structure:**
- Each file: `exercise_form_sampleN.npz`
- Contains: `landmarks` (n_frames × 132) and `label` metadata

In [ ]:
# Load all processed pose sequences
data_dir = Path('../data/processed')
files = sorted(data_dir.glob('*.npz'))

print(f"Found {len(files)} processed pose sequences")

# Organize data by exercise and form
data_dict = {
    'squats': {'correct': [], 'incorrect': []},
    'pushups': {'correct': [], 'incorrect': []},
    'bicep_curls': {'correct': [], 'incorrect': []}
}

for file in files:
    # Parse filename: exercise_form_sampleN.npz
    parts = file.stem.split('_')
    exercise = parts[0]
    form = parts[1]
    
    # Load data
    data = np.load(file)
    landmarks = data['landmarks']
    
    # Store
    if exercise in data_dict and form in data_dict[exercise]:
        data_dict[exercise][form].append(landmarks)

# Print summary
print("\nDataset Summary:")
print("=" * 60)
for exercise in data_dict:
    n_correct = len(data_dict[exercise]['correct'])
    n_incorrect = len(data_dict[exercise]['incorrect'])
    print(f"{exercise.upper():15} | Correct: {n_correct:3d} | Incorrect: {n_incorrect:3d} | Total: {n_correct+n_incorrect:3d}")

total_samples = sum(len(data_dict[ex]['correct']) + len(data_dict[ex]['incorrect']) for ex in data_dict)
print("=" * 60)
print(f"{'TOTAL':15} | Samples: {total_samples}")
print("\n✓ Data loaded successfully")

## 3. Angular Features: The Foundation of Pose Analysis

### WHY Joint Angles Are Critical

**Biomechanical Perspective:**
- Exercise quality = joint range of motion (ROM) and flexion/extension patterns
- Fitness professionals assess form using angles:
  - Squat: "Knees should bend to 90°"
  - Push-up: "Elbows should reach 90° at bottom"
  - Bicep curl: "Elbow flexion from 180° to 30°"

**Mathematical Advantages:**
1. **Rotation Invariance** 🔄: Angle between vectors remains constant regardless of coordinate system
2. **Scale Invariance** 📏: Person size doesn't affect angles
3. **Position Invariance** 📍: Camera location doesn't change joint angles

### Angle Calculation Formula

For three points P₁, P₂, P₃ forming an angle at P₂:

```
v₁ = P₁ - P₂  (vector from P₂ to P₁)
v₂ = P₃ - P₂  (vector from P₂ to P₃)

cos(θ) = (v₁ · v₂) / (|v₁| × |v₂|)
θ = arccos(cos(θ))  [in radians]
angle = θ × (180/π)  [in degrees]
```

### Angles We Extract (10 per frame)

| Angle | Points | Squat | Push-up | Bicep Curl |
|-------|--------|-------|---------|------------|
| Left Knee | Hip-Knee-Ankle | ⭐⭐⭐ | ⭐ | ⭐ |
| Right Knee | Hip-Knee-Ankle | ⭐⭐⭐ | ⭐ | ⭐ |
| Left Hip | Shoulder-Hip-Knee | ⭐⭐⭐ | ⭐ | ⭐ |
| Right Hip | Shoulder-Hip-Knee | ⭐⭐⭐ | ⭐ | ⭐ |
| Left Elbow | Shoulder-Elbow-Wrist | ⭐ | ⭐⭐⭐ | ⭐⭐⭐ |
| Right Elbow | Shoulder-Elbow-Wrist | ⭐ | ⭐⭐⭐ | ⭐⭐⭐ |
| Left Shoulder | Hip-Shoulder-Elbow | ⭐ | ⭐⭐ | ⭐⭐ |
| Right Shoulder | Hip-Shoulder-Elbow | ⭐ | ⭐⭐ | ⭐⭐ |
| Back Vertical | Shoulder-Hip-Vertical | ⭐⭐⭐ | ⭐ | ⭐ |
| Body Line | Ankle-Hip-Shoulder | ⭐ | ⭐⭐⭐ | ⭐ |

⭐⭐⭐ = Highly discriminative for this exercise

In [ ]:
# Initialize feature engineer
fe = FeatureEngineer()

# Extract angles from a sample squat sequence
sample_squat = data_dict['squats']['correct'][0]
print(f"Sample squat sequence shape: {sample_squat.shape}")
print(f"  → {sample_squat.shape[0]} frames")
print(f"  → {sample_squat.shape[1]} landmark values (33 landmarks × 4 values)\n")

# Extract angles for first frame
angles_frame_0 = fe.extract_angles(sample_squat[0])

print("Angles extracted from first frame:")
print("=" * 60)
for angle_name, angle_value in angles_frame_0.items():
    print(f"{angle_name:20} = {angle_value:6.1f}°")

print("\n✓ 10 angles extracted per frame")

### Visualize Angle Progression Through Exercise

In [ ]:
# Extract angles for entire sequence
angles_over_time = []
for frame in sample_squat:
    angles = fe.extract_angles(frame)
    angles_over_time.append(angles)

# Convert to DataFrame for easy plotting
angles_df = pd.DataFrame(angles_over_time)

# Plot key angles for squat
fig, axes = plt.subplots(2, 2, figsize=(15, 10))
fig.suptitle('Joint Angles Throughout Squat Movement', fontsize=16, fontweight='bold')

# Knee angles (most important for squats)
axes[0, 0].plot(angles_df['left_knee'], label='Left Knee', linewidth=2, marker='o', markersize=3)
axes[0, 0].plot(angles_df['right_knee'], label='Right Knee', linewidth=2, marker='s', markersize=3)
axes[0, 0].axhline(y=90, color='r', linestyle='--', label='Target: 90°', alpha=0.5)
axes[0, 0].set_xlabel('Frame')
axes[0, 0].set_ylabel('Angle (degrees)')
axes[0, 0].set_title('Knee Flexion Angles\n(Lower = Deeper Squat)')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# Hip angles
axes[0, 1].plot(angles_df['left_hip'], label='Left Hip', linewidth=2, marker='o', markersize=3)
axes[0, 1].plot(angles_df['right_hip'], label='Right Hip', linewidth=2, marker='s', markersize=3)
axes[0, 1].set_xlabel('Frame')
axes[0, 1].set_ylabel('Angle (degrees)')
axes[0, 1].set_title('Hip Flexion Angles\n(Lower = More Hip Hinge)')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# Back angle (posture)
axes[1, 0].plot(angles_df['back_vertical'], label='Back Alignment', linewidth=2, color='green', marker='d', markersize=3)
axes[1, 0].axhline(y=180, color='r', linestyle='--', label='Perfectly Upright', alpha=0.5)
axes[1, 0].set_xlabel('Frame')
axes[1, 0].set_ylabel('Angle (degrees)')
axes[1, 0].set_title('Back Vertical Alignment\n(Closer to 180° = Better Posture)')
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

# Body line angle
axes[1, 1].plot(angles_df['body_line'], label='Body Line', linewidth=2, color='purple', marker='^', markersize=3)
axes[1, 1].set_xlabel('Frame')
axes[1, 1].set_ylabel('Angle (degrees)')
axes[1, 1].set_title('Overall Body Alignment\n(Ankle-Hip-Shoulder)')
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n💡 Key Observations:")
print("   • Knee angle decreases as person squats down (lower = deeper)")
print("   • Symmetry between left/right indicates balanced form")
print("   • Back angle should remain stable (avoiding forward lean)")
print("   • These patterns differ drastically between exercises!")

## 4. Distance Features: Capturing Body Proportions

### WHY Distance Features Matter

**Complementary to Angles:**
- Angles tell us joint flexion, distances tell us body positioning
- Example: Two people can have same knee angle but different stance widths
- Stance width = knee-to-knee distance (critical for squat stability)

**Normalization Strategy:**
```
body_height = distance(mid_hip, mid_ankle)
normalized_distance = raw_distance / body_height
```

**Benefits:**
1. **Scale Invariance** 📏: Same normalized value for tall/short people
2. **Proportional Meaning** 📐: Values represent multiples of body height
3. **Cross-Person Comparability** 👥: Fair comparison across individuals

### Distance Features We Extract (10 per frame)

| Distance | Description | Exercise Relevance |
|----------|-------------|--------------------|
| **knee_width** | Knee-to-knee distance | Squat stance width |
| **ankle_width** | Ankle-to-ankle distance | Foot positioning |
| **shoulder_width** | Shoulder-to-shoulder | Body frame reference |
| **wrist_span** | Wrist-to-wrist distance | Arm extension (push-ups) |
| **left_elbow_hip** | Elbow-to-hip distance | Bicep curl form |
| **right_elbow_hip** | Elbow-to-hip distance | Bicep curl form |
| **left_arm_extension** | Wrist-to-shoulder | Arm straightness |
| **right_arm_extension** | Wrist-to-shoulder | Arm straightness |
| **hip_height** | Vertical hip position | Squat depth |
| **body_height_ref** | Hip-to-ankle distance | Normalization reference |

In [ ]:
# Extract distances from first frame
distances_frame_0 = fe.extract_distances(sample_squat[0])

print("Distance features extracted from first frame:")
print("=" * 60)
for dist_name, dist_value in distances_frame_0.items():
    if 'height' in dist_name or 'ref' in dist_name:
        print(f"{dist_name:25} = {dist_value:6.3f} (absolute units)")
    else:
        print(f"{dist_name:25} = {dist_value:6.3f} (× body height)")

print("\n💡 Interpretation:")
print(f"   • Body height reference: {distances_frame_0['body_height_ref']:.3f}")
print(f"   • Knee width = {distances_frame_0['knee_width']:.2f}× body height")
print(f"   • This means knees are {distances_frame_0['knee_width']:.1%} of body height apart")
print("\n✓ 10 distance features extracted per frame")

### Visualize Distance Features

In [ ]:
# Extract distances for entire sequence
distances_over_time = []
for frame in sample_squat:
    distances = fe.extract_distances(frame)
    distances_over_time.append(distances)

# Convert to DataFrame
distances_df = pd.DataFrame(distances_over_time)

# Plot key distances
fig, axes = plt.subplots(2, 2, figsize=(15, 10))
fig.suptitle('Body Distances Throughout Squat Movement', fontsize=16, fontweight='bold')

# Stance widths
axes[0, 0].plot(distances_df['knee_width'], label='Knee Width', linewidth=2, marker='o', markersize=3)
axes[0, 0].plot(distances_df['ankle_width'], label='Ankle Width', linewidth=2, marker='s', markersize=3)
axes[0, 0].set_xlabel('Frame')
axes[0, 0].set_ylabel('Distance (× body height)')
axes[0, 0].set_title('Stance Width Stability\n(Should Remain Constant)')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# Hip height (depth indicator)
axes[0, 1].plot(distances_df['hip_height'], linewidth=2, color='red', marker='d', markersize=3)
axes[0, 1].set_xlabel('Frame')
axes[0, 1].set_ylabel('Normalized Y coordinate')
axes[0, 1].set_title('Hip Height (Squat Depth)\n(Lower = Deeper Squat)')
axes[0, 1].grid(True, alpha=0.3)

# Arm positions
axes[1, 0].plot(distances_df['left_arm_extension'], label='Left Arm', linewidth=2, marker='o', markersize=3)
axes[1, 0].plot(distances_df['right_arm_extension'], label='Right Arm', linewidth=2, marker='s', markersize=3)
axes[1, 0].set_xlabel('Frame')
axes[1, 0].set_ylabel('Distance (× body height)')
axes[1, 0].set_title('Arm Extension\n(Wrist-to-Shoulder Distance)')
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

# Comparison: normalized vs. raw
knee_width_raw = []
for frame in sample_squat:
    left_knee = fe.get_landmark_coords(frame, 'left_knee')
    right_knee = fe.get_landmark_coords(frame, 'right_knee')
    from scipy.spatial.distance import euclidean
    raw_dist = euclidean(left_knee[:2], right_knee[:2])
    knee_width_raw.append(raw_dist)

ax2 = axes[1, 1].twinx()
axes[1, 1].plot(knee_width_raw, linewidth=2, color='blue', label='Raw Distance', marker='o', markersize=3)
ax2.plot(distances_df['knee_width'], linewidth=2, color='orange', label='Normalized', marker='s', markersize=3)
axes[1, 1].set_xlabel('Frame')
axes[1, 1].set_ylabel('Raw Distance', color='blue')
ax2.set_ylabel('Normalized (× body height)', color='orange')
axes[1, 1].set_title('Normalization Effect\n(Raw vs. Body-Height Normalized)')
axes[1, 1].legend(loc='upper left')
ax2.legend(loc='upper right')
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n💡 Key Observations:")
print("   • Stance width remains stable (good form)")
print("   • Hip height decreases during descent")
print("   • Normalization preserves pattern while removing scale dependency")

## 5. Temporal Features: Capturing Movement Dynamics

### WHY Temporal Features Are Essential

**Static vs. Dynamic Analysis:**
- Angles/distances = static snapshots
- Velocity/acceleration = how things change over time
- Exercise quality = both position AND speed of movement

**Real-World Examples:**
1. **Squat Speed** 🏋️
   - Slow, controlled descent = good form
   - Fast drop = poor control, injury risk
   - Velocity captures this temporal pattern

2. **Push-up Smoothness** 💪
   - Smooth, constant speed = strong core
   - Jerky, uneven motion = muscle fatigue
   - Acceleration spikes reveal instability

3. **Bicep Curl Tempo** 💪
   - Slow eccentric (lowering) = muscle building
   - Fast concentric (lifting) = momentum cheating
   - Velocity ratio distinguishes phases

### Mathematical Definitions

For position sequence $x[t]$:

**Velocity (1st derivative):**
```
v[t] = (x[t] - x[t-1]) / Δt
```
→ Rate of change of position (speed)

**Acceleration (2nd derivative):**
```
a[t] = (v[t] - v[t-1]) / Δt = (x[t] - 2x[t-1] + x[t-2]) / Δt²
```
→ Rate of change of velocity (force)

**Jerk (3rd derivative):**
```
j[t] = (a[t] - a[t-1]) / Δt
```
→ Rate of change of acceleration (smoothness)

**Smoothness Metric (LDLJ - Log Dimensionless Jerk):**
```
LDLJ = -log(∫ j² dt)
```
→ Lower jerk = smoother movement = better form

In [ ]:
# Extract temporal features
temporal_features = fe.extract_velocities(sample_squat)

print("Temporal features extracted:")
print("=" * 60)
print(f"Original sequence:    {sample_squat.shape[0]:3d} frames × {sample_squat.shape[1]:3d} features")
print(f"Velocities:           {temporal_features['velocities'].shape[0]:3d} frames × {temporal_features['velocities'].shape[1]:3d} features")
print(f"Accelerations:        {temporal_features['accelerations'].shape[0]:3d} frames × {temporal_features['accelerations'].shape[1]:3d} features")
print(f"Jerks:                {temporal_features['jerks'].shape[0]:3d} frames × {temporal_features['jerks'].shape[1]:3d} features")

print("\n💡 Note: Each differentiation reduces sequence length by 1")
print("   This is expected behavior for numerical derivatives")

# Calculate smoothness
smoothness = fe.calculate_smoothness(sample_squat)
print(f"\nMovement smoothness score: {smoothness:.3f}")
print("   (Higher = smoother, more controlled movement)")

### Visualize Velocity and Acceleration

In [ ]:
# Focus on hip vertical velocity (most important for squats)
# Hip landmark indices: left_hip=23, right_hip=24
# Y-coordinate is index 1 out of [x, y, z, visibility]
left_hip_idx = 23 * 4 + 1  # Y coordinate of left hip
right_hip_idx = 24 * 4 + 1  # Y coordinate of right hip

# Extract hip Y positions
hip_y_positions = (sample_squat[:, left_hip_idx] + sample_squat[:, right_hip_idx]) / 2

# Calculate velocity and acceleration manually for clarity
hip_velocity = np.diff(hip_y_positions)
hip_acceleration = np.diff(hip_velocity)
hip_jerk = np.diff(hip_acceleration)

# Create figure
fig, axes = plt.subplots(4, 1, figsize=(15, 12))
fig.suptitle('Temporal Derivatives: Hip Vertical Motion During Squat', fontsize=16, fontweight='bold')

# Position
axes[0].plot(hip_y_positions, linewidth=2, color='blue', marker='o', markersize=4)
axes[0].set_ylabel('Hip Y Position\n(normalized)', fontsize=11, fontweight='bold')
axes[0].set_title('Position: Raw Hip Height')
axes[0].grid(True, alpha=0.3)
axes[0].axhline(y=np.mean(hip_y_positions), color='r', linestyle='--', alpha=0.5, label='Mean')
axes[0].legend()

# Velocity
axes[1].plot(hip_velocity, linewidth=2, color='green', marker='s', markersize=4)
axes[1].set_ylabel('Velocity\n(Δy/frame)', fontsize=11, fontweight='bold')
axes[1].set_title('1st Derivative: Speed of Movement (Negative = Descending, Positive = Ascending)')
axes[1].axhline(y=0, color='black', linestyle='-', linewidth=1)
axes[1].grid(True, alpha=0.3)
axes[1].fill_between(range(len(hip_velocity)), hip_velocity, 0, where=(hip_velocity<0), alpha=0.3, color='red', label='Descending')
axes[1].fill_between(range(len(hip_velocity)), hip_velocity, 0, where=(hip_velocity>=0), alpha=0.3, color='blue', label='Ascending')
axes[1].legend()

# Acceleration
axes[2].plot(hip_acceleration, linewidth=2, color='orange', marker='^', markersize=4)
axes[2].set_ylabel('Acceleration\n(Δv/frame)', fontsize=11, fontweight='bold')
axes[2].set_title('2nd Derivative: Change in Speed (Peaks = Direction Changes)')
axes[2].axhline(y=0, color='black', linestyle='-', linewidth=1)
axes[2].grid(True, alpha=0.3)

# Jerk
axes[3].plot(hip_jerk, linewidth=2, color='red', marker='d', markersize=4)
axes[3].set_ylabel('Jerk\n(Δa/frame)', fontsize=11, fontweight='bold')
axes[3].set_xlabel('Frame Number', fontsize=12)
axes[3].set_title('3rd Derivative: Smoothness (Large Spikes = Jerky Motion)')
axes[3].axhline(y=0, color='black', linestyle='-', linewidth=1)
axes[3].grid(True, alpha=0.3)

# Highlight high jerk regions
jerk_threshold = np.std(hip_jerk) * 2
high_jerk_idx = np.where(np.abs(hip_jerk) > jerk_threshold)[0]
if len(high_jerk_idx) > 0:
    axes[3].scatter(high_jerk_idx, hip_jerk[high_jerk_idx], color='red', s=100, 
                    marker='x', linewidths=3, label='High Jerk (unstable)')
    axes[3].legend()

plt.tight_layout()
plt.show()

print("\n💡 Interpretation Guide:")
print("   📍 Position: Shows squat depth over time")
print("   🏃 Velocity: Negative = going down, positive = coming up")
print("   ⚡ Acceleration: Peaks at direction changes (bottom of squat)")
print("   📈 Jerk: Low values = smooth motion, spikes = sudden changes (bad form)")
print(f"\n   Smoothness score: {smoothness:.2f} (higher is better)")

## 6. Statistical Features: Aggregating Sequence Information

### WHY Statistical Aggregation?

**The Variable-Length Problem:**
- Different videos have different lengths (10 seconds vs. 30 seconds)
- Different rep counts (5 squats vs. 10 squats)
- Machine learning models need fixed-size inputs

**Solution: Statistical Summary**

For each feature (angles, distances) across all frames:

| Statistic | What It Captures | Example Interpretation |
|-----------|------------------|------------------------|
| **Mean** | Average position/angle | Overall squat depth |
| **Std Dev** | Variability/consistency | How much depth varies |
| **Min** | Extreme positions | Deepest squat point |
| **Max** | Extreme positions | Highest standing point |
| **Range** | Total excursion | Range of Motion (ROM) |

**Benefits:**
1. **Fixed-Size Representation** 📦: Variable sequences → Fixed 72 features
2. **Captures Patterns** 📊: Statistics encode movement characteristics
3. **Robust to Noise** 🛡️: Aggregation reduces frame-level noise

### Our Feature Engineering Pipeline

```
Raw Sequence (n_frames × 132)
       ↓
Extract Angles (n_frames × 10)
       ↓
Compute Statistics:
  - Mean (10 features)
  - Std (10 features)
  - Range (10 features)
       ↓
Angular Features: 30

Extract Distances (n_frames × 10)
       ↓
Compute Statistics:
  - Mean (10 features)
  - Std (10 features)
       ↓
Distance Features: 20

Velocities (first 10 dimensions)
  - Mean (10 features)
  - Max (10 features)
       ↓
Velocity Features: 20

Smoothness: 1
Sequence Length: 1
       ↓
TOTAL: 72 features
```

In [ ]:
# Extract complete feature vector using the pipeline
print("Engineering complete feature vector...\n")

features = fe.engineer_features(sample_squat)

print("Feature Engineering Complete!")
print("=" * 60)
print(f"Input:  Pose sequence with {sample_squat.shape[0]} frames × {sample_squat.shape[1]} landmarks")
print(f"Output: Fixed feature vector with {len(features)} features")
print("\nFeature Breakdown:")
print(f"  • Angular statistics (mean, std, range):  30 features")
print(f"  • Distance statistics (mean, std):        20 features")
print(f"  • Velocity statistics (mean, max):        20 features")
print(f"  • Smoothness metric:                       1 feature")
print(f"  • Sequence length:                         1 feature")
print(f"  {'─'*50}")
print(f"  TOTAL:                                    {len(features)} features")

print("\nSample feature values (first 10):")
print(features[:10])

print("\n✓ Dimensionality reduced: 132 → 72 (45% reduction)")
print("✓ Information preserved through domain knowledge")

## 7. Feature Visualization: Understanding Discriminative Power

### WHY Visualize Features?

Before training models, we need to understand:
1. **Do features separate exercises?** Can we visually distinguish squat vs. push-up?
2. **Do features separate correct vs. incorrect form?** Within same exercise?
3. **Which features are most discriminative?** To prioritize or select features
4. **Are features correlated?** To avoid redundancy

### What to Look For:
- ✅ **Clear separation** in distributions = good discriminative power
- ⚠️ **Overlapping distributions** = weak feature, may need engineering
- ✅ **Low correlation** = independent information
- ⚠️ **High correlation** = redundant features (consider removing)

In [ ]:
# Engineer features for all samples
print("Engineering features for entire dataset...")
print("This may take a minute...\n")

all_features = []
all_labels = []
all_exercises = []
all_forms = []

for exercise in data_dict:
    for form in data_dict[exercise]:
        for sequence in data_dict[exercise][form]:
            # Engineer features
            features = fe.engineer_features(sequence)
            
            # Store
            all_features.append(features)
            all_exercises.append(exercise)
            all_forms.append(form)
            all_labels.append(f"{exercise}_{form}")

# Convert to arrays
X = np.array(all_features)
y_exercise = np.array(all_exercises)
y_form = np.array(all_forms)
y_combined = np.array(all_labels)

print(f"✓ Engineered features for {len(X)} samples")
print(f"  Feature matrix shape: {X.shape}")
print(f"  ({X.shape[0]} samples × {X.shape[1]} features)")

### Feature Distribution by Exercise Type

In [ ]:
# Create feature names for reference
feature_names = (
    [f'angle_mean_{i}' for i in range(10)] +
    [f'angle_std_{i}' for i in range(10)] +
    [f'angle_range_{i}' for i in range(10)] +
    [f'dist_mean_{i}' for i in range(10)] +
    [f'dist_std_{i}' for i in range(10)] +
    [f'vel_mean_{i}' for i in range(10)] +
    [f'vel_max_{i}' for i in range(10)] +
    ['smoothness', 'seq_length']
)

# Create DataFrame for easy analysis
df_features = pd.DataFrame(X, columns=feature_names)
df_features['exercise'] = y_exercise
df_features['form'] = y_form
df_features['label'] = y_combined

# Plot distributions for key features
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('Feature Distributions by Exercise Type', fontsize=16, fontweight='bold')

# Select diverse features to showcase
key_features = [
    ('angle_mean_0', 'Mean Knee Angle'),
    ('angle_range_4', 'Elbow ROM'),
    ('dist_mean_0', 'Mean Knee Width'),
    ('vel_mean_0', 'Mean Velocity'),
    ('smoothness', 'Movement Smoothness'),
    ('seq_length', 'Sequence Length')
]

for idx, (feature, title) in enumerate(key_features):
    ax = axes[idx // 3, idx % 3]
    
    # Plot distributions for each exercise
    for exercise in ['squats', 'pushups', 'bicep_curls']:
        data = df_features[df_features['exercise'] == exercise][feature]
        ax.hist(data, alpha=0.5, label=exercise.replace('_', ' ').title(), bins=15, edgecolor='black')
    
    ax.set_xlabel(title, fontsize=11, fontweight='bold')
    ax.set_ylabel('Frequency', fontsize=11)
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n💡 Observations:")
print("   • Different exercises show distinct distributions")
print("   • Some features separate exercises better than others")
print("   • This validates our feature engineering approach!")

### Feature Correlation Heatmap

In [ ]:
# Compute correlation matrix (sample first 30 features for readability)
correlation_matrix = df_features.iloc[:, :30].corr()

# Plot heatmap
plt.figure(figsize=(14, 12))
sns.heatmap(correlation_matrix, 
            cmap='coolwarm', 
            center=0, 
            vmin=-1, vmax=1,
            square=True,
            linewidths=0.5,
            cbar_kws={'label': 'Correlation Coefficient'})
plt.title('Feature Correlation Matrix (First 30 Features)', fontsize=14, fontweight='bold', pad=20)
plt.xlabel('Feature Index', fontsize=12)
plt.ylabel('Feature Index', fontsize=12)
plt.tight_layout()
plt.show()

# Find highly correlated feature pairs
high_corr_pairs = []
for i in range(len(correlation_matrix.columns)):
    for j in range(i+1, len(correlation_matrix.columns)):
        if abs(correlation_matrix.iloc[i, j]) > 0.8:
            high_corr_pairs.append((
                correlation_matrix.columns[i],
                correlation_matrix.columns[j],
                correlation_matrix.iloc[i, j]
            ))

print(f"\nFound {len(high_corr_pairs)} highly correlated feature pairs (|r| > 0.8)")
if len(high_corr_pairs) > 0:
    print("\nTop correlated pairs:")
    for feat1, feat2, corr in sorted(high_corr_pairs, key=lambda x: abs(x[2]), reverse=True)[:5]:
        print(f"  • {feat1} ↔ {feat2}: {corr:.3f}")
    print("\n💡 Note: High correlation is expected (e.g., left/right symmetry)")
else:
    print("\n✓ No highly correlated pairs found - features are independent!")

### Feature Importance Preview (Using Random Forest)

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder

# Encode labels
le = LabelEncoder()
y_encoded = le.fit_transform(y_exercise)

# Train a quick Random Forest
rf = RandomForestClassifier(n_estimators=100, random_state=42, max_depth=10)
rf.fit(X, y_encoded)

# Get feature importances
importances = rf.feature_importances_
indices = np.argsort(importances)[::-1]

# Plot top 20 features
plt.figure(figsize=(14, 8))
top_n = 20
top_indices = indices[:top_n]
top_importances = importances[top_indices]
top_names = [feature_names[i] for i in top_indices]

colors = ['red' if 'angle' in name else 'blue' if 'dist' in name else 'green' if 'vel' in name else 'purple' 
          for name in top_names]

bars = plt.barh(range(top_n), top_importances, color=colors, edgecolor='black', linewidth=1.2)
plt.yticks(range(top_n), top_names)
plt.xlabel('Feature Importance (Random Forest)', fontsize=12, fontweight='bold')
plt.ylabel('Feature Name', fontsize=12, fontweight='bold')
plt.title('Top 20 Most Important Features for Exercise Classification', fontsize=14, fontweight='bold')
plt.gca().invert_yaxis()

# Add legend
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='red', edgecolor='black', label='Angular Features'),
    Patch(facecolor='blue', edgecolor='black', label='Distance Features'),
    Patch(facecolor='green', edgecolor='black', label='Velocity Features'),
    Patch(facecolor='purple', edgecolor='black', label='Other Features')
]
plt.legend(handles=legend_elements, loc='lower right')

plt.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

print("\n💡 Feature Importance Insights:")
print(f"   • Top feature: {top_names[0]} (importance: {top_importances[0]:.4f})")
print(f"   • Angular features in top 10: {sum(1 for n in top_names[:10] if 'angle' in n)}")
print(f"   • Distance features in top 10: {sum(1 for n in top_names[:10] if 'dist' in n)}")
print(f"   • Velocity features in top 10: {sum(1 for n in top_names[:10] if 'vel' in n)}")
print("\n   → Mix of feature types confirms our multi-modal approach is valuable!")

## 8. Save Engineered Features

**WHY save to separate directory?**
- Separate raw data from processed features
- Enable fast model training (skip feature engineering step)
- Version control for feature engineering changes
- Easy to load in model training notebooks

**Output Structure:**
```
data/features/
├── features.npy        # Feature matrix (n_samples × 72)
├── labels_exercise.npy # Exercise labels (squats/pushups/curls)
├── labels_form.npy     # Form labels (correct/incorrect)
├── labels_combined.npy # Combined labels (exercise_form)
└── feature_names.txt   # Feature documentation
```

In [ ]:
# Create features directory
features_dir = Path('../data/features')
features_dir.mkdir(parents=True, exist_ok=True)

# Save features and labels
np.save(features_dir / 'features.npy', X)
np.save(features_dir / 'labels_exercise.npy', y_exercise)
np.save(features_dir / 'labels_form.npy', y_form)
np.save(features_dir / 'labels_combined.npy', y_combined)

# Save feature names
with open(features_dir / 'feature_names.txt', 'w') as f:
    f.write("Feature Engineering Documentation\n")
    f.write("=" * 60 + "\n\n")
    f.write(f"Total Features: {len(feature_names)}\n")
    f.write(f"Total Samples: {len(X)}\n\n")
    
    f.write("Feature Breakdown:\n")
    f.write("-" * 60 + "\n")
    f.write("Angular Features (30):\n")
    f.write("  - Mean angles (10): Average joint angles across sequence\n")
    f.write("  - Std angles (10): Variability of joint angles\n")
    f.write("  - Range angles (10): Range of Motion (ROM)\n\n")
    
    f.write("Distance Features (20):\n")
    f.write("  - Mean distances (10): Average inter-joint distances\n")
    f.write("  - Std distances (10): Variability of distances\n\n")
    
    f.write("Velocity Features (20):\n")
    f.write("  - Mean velocities (10): Average movement speed\n")
    f.write("  - Max velocities (10): Peak movement speed\n\n")
    
    f.write("Temporal Features (2):\n")
    f.write("  - Smoothness (1): Jerk-based movement quality\n")
    f.write("  - Sequence length (1): Number of frames\n\n")
    
    f.write("Feature Names:\n")
    f.write("-" * 60 + "\n")
    for i, name in enumerate(feature_names, 1):
        f.write(f"{i:3d}. {name}\n")

print("✓ Features saved successfully!\n")
print("Saved files:")
for file in sorted(features_dir.glob('*')):
    size = file.stat().st_size
    print(f"  • {file.name:25} ({size:,} bytes)")

print(f"\nLocation: {features_dir.absolute()}")

## 9. Summary: Feature Engineering Impact

### What We Accomplished

We transformed raw pose landmarks into biomechanically meaningful features:

| Aspect | Before | After | Improvement |
|--------|--------|-------|-------------|
| **Dimensionality** | 132 features | 72 features | 45% reduction |
| **Interpretability** | Raw x,y,z coords | Angles, distances, velocities | ✅ Domain meaning |
| **Invariance** | Position/scale dependent | Normalized & invariant | ✅ Robust |
| **Sequence Length** | Variable (n_frames × 132) | Fixed (72) | ✅ ML-ready |
| **Information** | Redundant coordinates | Discriminative features | ✅ Signal-rich |

---

### Feature Categories Explained

#### 1️⃣ Angular Features (30 features)

**WHY:** Angles are the gold standard in biomechanics
- **Rotation invariant**: Same angle regardless of camera viewpoint
- **Scale invariant**: Same angle for tall/short people
- **Interpretable**: Fitness professionals think in angles
- **Discriminative**: Each exercise has unique angle patterns

**Examples:**
- Squat: Knee angle 90° at bottom position
- Push-up: Elbow angle 90° at bottom position
- Bicep curl: Elbow flexion from 180° to 30°

---

#### 2️⃣ Distance Features (20 features)

**WHY:** Distances capture body positioning and proportions
- **Normalized by body height**: Removes scale dependency
- **Stance analysis**: Knee width, ankle width for balance
- **Arm positioning**: Wrist span, arm extension
- **Form detection**: Elbow-hip distance for curl form

**Examples:**
- Squat stance width = 0.3× body height
- Push-up wrist span = 1.2× shoulder width
- Curl elbow position = fixed at hips

---

#### 3️⃣ Velocity Features (20 features)

**WHY:** Movement dynamics distinguish exercise quality
- **Speed patterns**: Controlled vs. rushed movements
- **Phase detection**: Concentric vs. eccentric phases
- **Tempo analysis**: Fast descent = poor form
- **Momentum detection**: Using swing vs. muscle control

**Examples:**
- Proper squat: Slow descent, controlled ascent
- Poor curl: Fast concentric (using momentum)
- Good push-up: Consistent velocity throughout

---

#### 4️⃣ Smoothness (1 feature)

**WHY:** Jerk distinguishes controlled vs. unstable movement
- **Lower jerk = smoother**: Indicates muscle control
- **Higher jerk = jerky**: Suggests poor form or fatigue
- **Dimensionless**: Comparable across exercises
- **Scientifically validated**: Used in motor control research

**Formula:** LDLJ = -log(∫ jerk² dt)

---

#### 5️⃣ Temporal (1 feature)

**WHY:** Exercise tempo affects difficulty and effectiveness
- **Sequence length**: Number of frames in video
- **Indirect tempo**: Longer sequences = slower movements
- **Rep count indicator**: More reps = longer sequence
- **Fatigue detection**: Later reps often slower

---

### Why This Approach Works

✅ **Domain Knowledge Integration**
- We're not just applying ML blindly
- Features encode fitness expertise
- Model learns meaningful patterns, not coincidences

✅ **Invariance Properties**
- Robust to camera position, zoom, lighting
- Generalizes across different people, locations
- Focuses on movement, not appearance

✅ **Dimensionality Reduction**
- 72 features vs. 132 raw landmarks
- Faster training, less overfitting
- Each feature carries more signal

✅ **Interpretability**
- Can explain model predictions
- "Knee angle too high" vs. "Feature #47 is wrong"
- Builds trust with users

---

### Next Steps

With engineered features ready, we can now:

1. **Model Training** 🤖
   - Train classifiers on 72 features (not 132)
   - Compare ML algorithms (RF, SVM, Neural Networks)
   - Evaluate on test set

2. **Form Correction** 📋
   - Identify which features deviate from correct form
   - Provide specific feedback ("Knees caving in")
   - Track improvement over time

3. **Deployment** 🚀
   - Real-time feature extraction from webcam
   - Instant exercise classification
   - Live form feedback

---

## Key Takeaway

> **Feature engineering is where domain expertise meets machine learning.**
>
> By transforming raw pose landmarks into biomechanically meaningful features, we:
> - Make models more accurate
> - Faster to train
> - Easier to interpret
> - More robust to variations
>
> **This is the difference between "doing ML" and "doing ML right."** 💪

---

**End of Feature Engineering Notebook** ✅